# penuX — Pathogen Prediction from Inpatient Vital Signs

Adapted from [netanelcyber/penuX](https://github.com/netanelcyber/penuX) for the Parthenon Morpheus workbench.

**What this does:** Predicts pathogen class (Normal / Bacterial / Viral) from four clinical vital signs — temperature, white blood cell count, SpO2, and age — using a neural network trained on MIMIC data.

**This notebook:**
1. **Connect** — database setup, extract features live from the `mimiciv` schema
2. **Feature extraction** — build a clinical dataset (temperature, WBC, SpO2, age, pathogen label) from chartevents, labevents, microbiologyevents
3. **Correlation analysis** — Pearson & Spearman matrices, grouped by pathogen class
4. **Model training** — train the penuX neural network on extracted features
5. **Prediction** — classify pathogen type from vitals
6. **Calibration & evaluation** — ECE analysis, reliability diagrams, ROC curves, subgroup bias

---

**Research use only.** Not intended for clinical decision-making. Based on de-identified MIMIC data.

## 1. Environment & Connection Setup

In [ ]:
from __future__ import annotations

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

NOTEBOOK_DIR = Path(os.environ.get('PARTHENON_NOTEBOOK_DIR', '/home/jovyan/notebooks'))
OUTPUT_DIR = NOTEBOOK_DIR / 'output' / 'penux'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Database ──
DB_USER = os.environ.get('PARTHENON_DB_USER', 'parthenon')
DB_PASS = os.environ.get('PARTHENON_DB_PASSWORD', '')
DB_HOST = os.environ.get('PARTHENON_DB_HOST', 'postgres')
DB_PORT = os.environ.get('PARTHENON_DB_PORT', '5432')
DB_NAME = os.environ.get('PARTHENON_DB_NAME', 'parthenon')

DATABASE_URL = f'postgresql+psycopg://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(DATABASE_URL)

# ── Schema (configurable for multi-dataset support) ──
SCHEMA = os.environ.get('MORPHEUS_SCHEMA', 'mimiciv')

def query(sql: str, params: dict | None = None) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

# ── Verify ──
with engine.connect() as conn:
    row = conn.execute(text('SELECT current_user, current_database()')).fetchone()
    print(f'\u2713 Connected to {row[1]} as {row[0]}')
    print(f'\u2713 Schema: {SCHEMA}')
    print(f'\u2713 Output: {OUTPUT_DIR}')

## 2. Feature Extraction from MIMIC-IV

Build the penuX clinical dataset by joining:
- **Temperature** (\u00b0C) from `chartevents` (itemid 223762)
- **SpO2** (%) from `chartevents` (itemid 220277)
- **WBC** (10\u00b3/\u00b5L) from `labevents` (label ILIKE '%white blood%')
- **Age** from `patients.anchor_age`
- **Pathogen label** from `microbiologyevents.org_name`

In [ ]:
# ── Step 2a: Identify admissions with positive microbiology cultures ──
micro = query(f"""
    SELECT DISTINCT m.subject_id, m.hadm_id, m.chartdate, m.org_name,
           m.spec_type_desc, m.test_name
    FROM {SCHEMA}.microbiologyevents m
    WHERE m.org_name IS NOT NULL
      AND m.hadm_id IS NOT NULL
    ORDER BY m.subject_id, m.chartdate
""")
print(f'Microbiology cultures with organisms: {len(micro):,}')
print(f'Unique patients: {micro["subject_id"].nunique():,}')
print(f'Unique organisms: {micro["org_name"].nunique()}')
print()
micro['org_name'].value_counts().head(15)

In [ ]:
# ── Step 2b: Map organisms to penuX pathogen classes ──

# penuX 3-class mapping: Normal (no positive culture), Bacterial, Viral
# Extended pathogen vocab for species-level prediction
PATHOGEN_MAP = {
    # Bacterial
    'PSEUDOMONAS AERUGINOSA': 'Bacterial',
    'ESCHERICHIA COLI': 'Bacterial',
    'PROTEUS MIRABILIS': 'Bacterial',
    'STAPH AUREUS COAG +': 'Bacterial',
    'STAPHYLOCOCCUS AUREUS': 'Bacterial',
    'STAPHYLOCOCCUS, COAGULASE NEGATIVE': 'Bacterial',
    'KLEBSIELLA PNEUMONIAE': 'Bacterial',
    'ENTEROCOCCUS SP.': 'Bacterial',
    'ENTEROCOCCUS FAECALIS': 'Bacterial',
    'ENTEROCOCCUS FAECIUM': 'Bacterial',
    'SERRATIA MARCESCENS': 'Bacterial',
    'CITROBACTER FREUNDII COMPLEX': 'Bacterial',
    'ENTEROBACTER CLOACAE COMPLEX': 'Bacterial',
    'ACINETOBACTER BAUMANNII': 'Bacterial',
    'STENOTROPHOMONAS MALTOPHILIA': 'Bacterial',
    'POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS': 'Bacterial',
    # Fungal (classified as Bacterial in penuX binary/3-class)
    'YEAST': 'Bacterial',
    'CANDIDA ALBICANS': 'Bacterial',
    'CANDIDA GLABRATA': 'Bacterial',
}

def classify_organism(org_name: str) -> str:
    """Map organism name to penuX 3-class label."""
    if org_name is None:
        return 'Normal'
    name = org_name.upper().strip()
    if name in PATHOGEN_MAP:
        return PATHOGEN_MAP[name]
    # Heuristic: most MIMIC microbiology isolates are bacterial
    if any(kw in name for kw in ['VIRUS', 'VIRAL', 'RSV', 'INFLUENZA', 'COVID', 'PARAINFLUENZA']):
        return 'Viral'
    return 'Bacterial'

micro['label'] = micro['org_name'].apply(classify_organism)
print('Label distribution:')
micro['label'].value_counts()

In [ ]:
# ── Step 2c: Extract temperature and SpO2 from chartevents ──
# MIMIC-IV item IDs: Temperature (C) = 223762, SpO2 = 220277

# Get admission-level aggregated vitals for micro-positive admissions
hadm_ids = micro['hadm_id'].unique().tolist()
print(f'Fetching vitals for {len(hadm_ids)} admissions...')

# Process in batches to avoid overly large IN clauses
BATCH_SIZE = 500
vitals_frames = []

for i in range(0, len(hadm_ids), BATCH_SIZE):
    batch = hadm_ids[i:i + BATCH_SIZE]
    placeholders = ','.join(str(int(h)) for h in batch)
    df = query(f"""
        SELECT c.hadm_id,
               MAX(CASE WHEN c.itemid = 223762 THEN c.valuenum END) AS temperature_c,
               MAX(CASE WHEN c.itemid = 220277 THEN c.valuenum END) AS spo2
        FROM {SCHEMA}.chartevents c
        WHERE c.hadm_id IN ({placeholders})
          AND c.itemid IN (223762, 220277)
          AND c.valuenum IS NOT NULL
        GROUP BY c.hadm_id
    """)
    vitals_frames.append(df)

vitals = pd.concat(vitals_frames, ignore_index=True) if vitals_frames else pd.DataFrame()
print(f'Vitals extracted: {len(vitals)} admissions')
vitals.describe()

In [ ]:
# ── Step 2d: Extract WBC from labevents ──

wbc_frames = []

for i in range(0, len(hadm_ids), BATCH_SIZE):
    batch = hadm_ids[i:i + BATCH_SIZE]
    placeholders = ','.join(str(int(h)) for h in batch)
    df = query(f"""
        SELECT l.hadm_id,
               MAX(l.valuenum) AS wbc
        FROM {SCHEMA}.labevents l
        JOIN {SCHEMA}.d_labitems di ON l.itemid = di.itemid
        WHERE l.hadm_id IN ({placeholders})
          AND di.label ILIKE '%white blood cell%'
          AND l.valuenum IS NOT NULL
          AND l.valuenum BETWEEN 0.1 AND 500  -- Filter implausible values
        GROUP BY l.hadm_id
    """)
    wbc_frames.append(df)

wbc_df = pd.concat(wbc_frames, ignore_index=True) if wbc_frames else pd.DataFrame()
print(f'WBC extracted: {len(wbc_df)} admissions')
wbc_df.describe()

In [ ]:
# ── Step 2e: Get patient demographics (age) ──

subject_ids = micro['subject_id'].unique().tolist()
placeholders = ','.join(str(int(s)) for s in subject_ids)

demographics = query(f"""
    SELECT subject_id, anchor_age AS age, gender
    FROM {SCHEMA}.patients
    WHERE subject_id IN ({placeholders})
""")
print(f'Demographics: {len(demographics)} patients')

In [ ]:
# ── Step 2f: Assemble the penuX clinical dataset ──

# Take the first positive culture per admission
micro_dedup = micro.sort_values('chartdate').drop_duplicates(subset=['hadm_id'], keep='first')

# Merge everything on hadm_id / subject_id
clinical = (
    micro_dedup[['subject_id', 'hadm_id', 'org_name', 'label']]
    .merge(vitals, on='hadm_id', how='inner')
    .merge(wbc_df, on='hadm_id', how='inner')
    .merge(demographics[['subject_id', 'age', 'gender']], on='subject_id', how='inner')
)

# Drop rows with missing vitals
clinical = clinical.dropna(subset=['temperature_c', 'wbc', 'spo2', 'age'])

# WBC: if stored in 10^3/uL (typical MIMIC), convert to cells/uL for penuX
# penuX expects values like 7000, 15000 — if median < 100, multiply by 1000
if clinical['wbc'].median() < 100:
    clinical['wbc'] = clinical['wbc'] * 1000

print(f'\u2713 Clinical dataset assembled: {len(clinical)} rows')
print(f'  Features: temperature_c, wbc, spo2, age')
print(f'  Labels: {clinical["label"].value_counts().to_dict()}')
print()
clinical[['temperature_c', 'wbc', 'spo2', 'age', 'label']].describe()

In [ ]:
# Add "Normal" class from admissions WITHOUT positive cultures

positive_hadms = set(micro['hadm_id'].unique())

normals = query(f"""
    WITH vital_agg AS (
        SELECT c.hadm_id,
               MAX(CASE WHEN c.itemid = 223762 THEN c.valuenum END) AS temperature_c,
               MAX(CASE WHEN c.itemid = 220277 THEN c.valuenum END) AS spo2
        FROM {SCHEMA}.chartevents c
        WHERE c.itemid IN (223762, 220277)
          AND c.valuenum IS NOT NULL
        GROUP BY c.hadm_id
    ),
    wbc_agg AS (
        SELECT l.hadm_id, MAX(l.valuenum) AS wbc
        FROM {SCHEMA}.labevents l
        JOIN {SCHEMA}.d_labitems di ON l.itemid = di.itemid
        WHERE di.label ILIKE '%white blood cell%'
          AND l.valuenum IS NOT NULL
          AND l.valuenum BETWEEN 0.1 AND 500
        GROUP BY l.hadm_id
    )
    SELECT a.subject_id, a.hadm_id, p.anchor_age AS age, p.gender,
           v.temperature_c, v.spo2, w.wbc
    FROM {SCHEMA}.admissions a
    JOIN {SCHEMA}.patients p ON a.subject_id = p.subject_id
    JOIN vital_agg v ON a.hadm_id = v.hadm_id
    JOIN wbc_agg w ON a.hadm_id = w.hadm_id
    WHERE a.hadm_id NOT IN (
        SELECT DISTINCT hadm_id FROM {SCHEMA}.microbiologyevents
        WHERE org_name IS NOT NULL AND hadm_id IS NOT NULL
    )
    AND v.temperature_c IS NOT NULL
    AND v.spo2 IS NOT NULL
    AND w.wbc IS NOT NULL
    ORDER BY RANDOM()
    LIMIT {len(clinical)}  -- Balance with positive class size
""")

normals['label'] = 'Normal'
normals['org_name'] = None

if normals['wbc'].median() < 100:
    normals['wbc'] = normals['wbc'] * 1000

# Combine
dataset = pd.concat([clinical, normals], ignore_index=True)
dataset = dataset.dropna(subset=['temperature_c', 'wbc', 'spo2', 'age'])

print(f'\u2713 Full dataset: {len(dataset)} rows')
print(f'  Label distribution:')
dataset['label'].value_counts()

In [ ]:
# Save the extracted dataset
dataset[['temperature_c', 'wbc', 'spo2', 'age', 'label', 'org_name', 'gender']].to_csv(
    OUTPUT_DIR / 'clinical_dataset.csv', index=False
)
print(f'\u2713 Saved to {OUTPUT_DIR / "clinical_dataset.csv"}')

## 3. Correlation Analysis

Compute Pearson and Spearman correlation matrices on vitals, grouped by pathogen class. Reproduces penuX `compute_correlations.py` functionality.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')

COLORS = {
    'bg': '#0E0E11',
    'panel': '#18181B',
    'border': '#27272A',
    'text': '#C5C0B8',
    'crimson': '#9B1B30',
    'gold': '#C9A227',
    'teal': '#2DD4BF',
    'muted': '#71717A',
}

def penux_fig(width=10, height=6, title=''):
    fig, ax = plt.subplots(figsize=(width, height), facecolor=COLORS['bg'])
    ax.set_facecolor(COLORS['bg'])
    ax.tick_params(colors=COLORS['text'], labelsize=9)
    for spine in ax.spines.values():
        spine.set_color(COLORS['border'])
    if title:
        ax.set_title(title, color=COLORS['text'], fontsize=14, fontweight='bold', pad=12)
    return fig, ax

print('\u2713 Visualization ready')

In [ ]:
FEATURES = ['temperature_c', 'wbc', 'spo2', 'age']
FEATURE_LABELS = ['Temp (\u00b0C)', 'WBC', 'SpO2 (%)', 'Age']

# ── Overall correlation ──
pearson = dataset[FEATURES].corr(method='pearson')
spearman = dataset[FEATURES].corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor=COLORS['bg'])

for ax, corr, method in zip(axes, [pearson, spearman], ['Pearson', 'Spearman']):
    ax.set_facecolor(COLORS['bg'])
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, ax=ax, square=True,
                xticklabels=FEATURE_LABELS, yticklabels=FEATURE_LABELS,
                cbar_kws={'shrink': 0.8})
    ax.set_title(f'{method} Correlation', color=COLORS['text'], fontsize=13, fontweight='bold')
    ax.tick_params(colors=COLORS['text'])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'correlation_overall.png', dpi=150, bbox_inches='tight',
            facecolor=COLORS['bg'])
plt.show()

pearson.to_csv(OUTPUT_DIR / 'pearson_overall.csv')
spearman.to_csv(OUTPUT_DIR / 'spearman_overall.csv')
print('\u2713 Saved correlation matrices')

In [ ]:
# ── Per-group correlation (by pathogen class) ──
labels = dataset['label'].unique()
n_groups = len(labels)

fig, axes = plt.subplots(1, n_groups, figsize=(6 * n_groups, 5), facecolor=COLORS['bg'])
if n_groups == 1:
    axes = [axes]

for ax, label in zip(axes, sorted(labels)):
    subset = dataset[dataset['label'] == label][FEATURES]
    corr = subset.corr(method='pearson')
    ax.set_facecolor(COLORS['bg'])
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, ax=ax, square=True,
                xticklabels=FEATURE_LABELS, yticklabels=FEATURE_LABELS)
    ax.set_title(f'{label} (n={len(subset)})', color=COLORS['text'], fontsize=12, fontweight='bold')
    ax.tick_params(colors=COLORS['text'])

plt.suptitle('Pearson Correlation by Pathogen Class', color=COLORS['text'],
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'correlation_by_class.png', dpi=150, bbox_inches='tight',
            facecolor=COLORS['bg'])
plt.show()

## 4. Model Training

Train the penuX neural network — a feedforward clinical encoder that predicts pathogen class from 4 vitals.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix

# ── Prepare data ──
X = dataset[FEATURES].values.astype(np.float32)
le = LabelEncoder()
y = le.fit_transform(dataset['label'])
class_names = le.classes_
n_classes = len(class_names)

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/val/test split
X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f'Classes: {list(class_names)}')
print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

# Save scaler for inference
np.savez(OUTPUT_DIR / 'clin_scaler.npz', mean=scaler.mean_, scale=scaler.scale_)

In [ ]:
# ── Build the penuX clinical encoder ──
# Architecture: Dense(64,relu) -> BatchNorm -> Dense(32,relu) -> Dense(n_classes,softmax)

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers

    tf.random.set_seed(42)

    model = keras.Sequential([
        layers.Input(shape=(4,)),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(n_classes, activation='softmax'),
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )

    model.summary()
    USE_TF = True

except ImportError:
    print('TensorFlow not available — falling back to scikit-learn MLPClassifier')
    USE_TF = False

In [ ]:
# ── Train ──

if USE_TF:
    # One-hot for AUC monitoring
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=32,
        callbacks=[early_stop],
        verbose=1,
    )

    # Save model
    model.save(OUTPUT_DIR / 'clin_encoder.keras')
    print(f'\u2713 Model saved to {OUTPUT_DIR / "clin_encoder.keras"}')

else:
    from sklearn.neural_network import MLPClassifier

    model = MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        max_iter=200,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.15,
    )
    model.fit(X_train, y_train)
    print(f'\u2713 MLPClassifier trained (best val score: {model.best_validation_score_:.3f})')

In [ ]:
# ── Training curves (TensorFlow only) ──

if USE_TF and history:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=COLORS['bg'])

    for ax, metric, label in zip(axes, ['loss', 'accuracy'], ['Loss', 'Accuracy']):
        ax.set_facecolor(COLORS['bg'])
        ax.plot(history.history[metric], color=COLORS['teal'], label='Train')
        ax.plot(history.history[f'val_{metric}'], color=COLORS['crimson'], label='Val')
        ax.set_xlabel('Epoch', color=COLORS['muted'])
        ax.set_ylabel(label, color=COLORS['muted'])
        ax.set_title(f'Training {label}', color=COLORS['text'], fontsize=13, fontweight='bold')
        ax.legend(facecolor=COLORS['panel'], edgecolor=COLORS['border'])
        for spine in ax.spines.values():
            spine.set_color(COLORS['border'])
        ax.tick_params(colors=COLORS['text'])

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight',
                facecolor=COLORS['bg'])
    plt.show()

## 5. Prediction

Classify pathogen type from vitals — single patient or batch.

In [ ]:
def predict_pathogen(temperature_c: float, wbc: float, spo2: float, age: float,
                     top_k: int | None = None) -> pd.DataFrame:
    """Predict pathogen class from clinical vitals.

    Args:
        temperature_c: Body temperature in Celsius
        wbc: White blood cell count (cells/uL, e.g. 7000)
        spo2: Oxygen saturation (%)
        age: Patient age in years
        top_k: Return only top-K most likely classes
    """
    x = np.array([[temperature_c, wbc, spo2, age]], dtype=np.float32)
    x_scaled = scaler.transform(x)

    if USE_TF:
        probs = model.predict(x_scaled, verbose=0)[0]
    else:
        probs = model.predict_proba(x_scaled)[0]

    result = pd.DataFrame({
        'class': class_names,
        'probability': probs,
    }).sort_values('probability', ascending=False)

    if top_k:
        result = result.head(top_k)

    return result.reset_index(drop=True)


def predict_batch(df: pd.DataFrame, top_k: int | None = None) -> pd.DataFrame:
    """Predict pathogen class for a batch of patients.

    Args:
        df: DataFrame with columns: temperature_c, wbc, spo2, age
        top_k: Include only top-K classes per patient
    """
    x = df[FEATURES].values.astype(np.float32)
    x_scaled = scaler.transform(x)

    if USE_TF:
        probs = model.predict(x_scaled, verbose=0)
    else:
        probs = model.predict_proba(x_scaled)

    prob_df = pd.DataFrame(probs, columns=class_names, index=df.index)
    prob_df['predicted'] = class_names[probs.argmax(axis=1)]
    prob_df['confidence'] = probs.max(axis=1)

    return pd.concat([df[FEATURES], prob_df], axis=1)

print('\u2713 Prediction functions ready')

In [ ]:
# ── Single patient prediction ──
print('=== Example: High fever + high WBC (suggests bacterial) ===')
display(predict_pathogen(temperature_c=39.0, wbc=18000, spo2=93, age=62))

print()
print('=== Example: Moderate fever + normal WBC (suggests viral) ===')
display(predict_pathogen(temperature_c=38.2, wbc=6500, spo2=95, age=45))

print()
print('=== Example: Normal vitals ===')
display(predict_pathogen(temperature_c=36.8, wbc=7000, spo2=98, age=35))

In [ ]:
# ── Batch prediction on test set ──
test_df = pd.DataFrame(X_test, columns=FEATURES)
# Inverse-transform to original scale for readability
test_df[FEATURES] = scaler.inverse_transform(X_test)
test_df['true_label'] = le.inverse_transform(y_test)

results = predict_batch(test_df)
print(f'Test set predictions: {len(results)} patients')
results.head(10)

## 6. Calibration & Evaluation

Evaluate the model with metrics from penuX: classification report, ROC curves, Expected Calibration Error (ECE), reliability diagrams, and subgroup bias analysis.

In [ ]:
# ── Test set predictions ──
if USE_TF:
    y_probs = model.predict(X_test, verbose=0)
else:
    y_probs = model.predict_proba(X_test)

y_pred = y_probs.argmax(axis=1)

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
# ── Confusion Matrix ──
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 6), facecolor=COLORS['bg'])
ax.set_facecolor(COLORS['bg'])
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
            xticklabels=class_names, yticklabels=class_names)
ax.set_xlabel('Predicted', color=COLORS['text'])
ax.set_ylabel('Actual', color=COLORS['text'])
ax.set_title('Confusion Matrix', color=COLORS['text'], fontsize=14, fontweight='bold')
ax.tick_params(colors=COLORS['text'])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight',
            facecolor=COLORS['bg'])
plt.show()

In [ ]:
# ── ROC Curves (one-vs-rest) ──
from sklearn.preprocessing import label_binarize

y_test_bin = label_binarize(y_test, classes=range(n_classes))

fig, ax = penux_fig(9, 6, 'ROC Curves (One-vs-Rest)')
colors_cycle = [COLORS['teal'], COLORS['crimson'], COLORS['gold'], '#818CF8']

for i, (cls, color) in enumerate(zip(class_names, colors_cycle)):
    if n_classes == 2 and i == 0:
        continue  # Skip redundant class in binary
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_probs[:, i])
    auc = roc_auc_score(y_test_bin[:, i], y_probs[:, i])
    ax.plot(fpr, tpr, color=color, linewidth=1.5, label=f'{cls} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], '--', color=COLORS['muted'], linewidth=0.8)
ax.set_xlabel('False Positive Rate', color=COLORS['muted'])
ax.set_ylabel('True Positive Rate', color=COLORS['muted'])
ax.legend(facecolor=COLORS['panel'], edgecolor=COLORS['border'])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'roc_curves.png', dpi=150, bbox_inches='tight',
            facecolor=COLORS['bg'])
plt.show()

In [ ]:
# ── Expected Calibration Error (ECE) & Reliability Diagram ──
# Key penuX metric: measures how well predicted probabilities match actual outcomes

def compute_ece(y_true: np.ndarray, y_probs: np.ndarray, n_bins: int = 10) -> tuple[float, list]:
    """Compute Expected Calibration Error."""
    confidences = y_probs.max(axis=1)
    predictions = y_probs.argmax(axis=1)
    accuracies = (predictions == y_true).astype(float)

    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bins_data = []

    for i in range(n_bins):
        mask = (confidences > bin_edges[i]) & (confidences <= bin_edges[i + 1])
        if mask.sum() == 0:
            bins_data.append({'bin_center': (bin_edges[i] + bin_edges[i + 1]) / 2,
                              'accuracy': 0, 'confidence': 0, 'count': 0})
            continue

        bin_acc = accuracies[mask].mean()
        bin_conf = confidences[mask].mean()
        bin_count = mask.sum()
        ece += (bin_count / len(y_true)) * abs(bin_acc - bin_conf)
        bins_data.append({'bin_center': (bin_edges[i] + bin_edges[i + 1]) / 2,
                          'accuracy': bin_acc, 'confidence': bin_conf, 'count': bin_count})

    return ece, bins_data


ece, bins_data = compute_ece(y_test, y_probs)
bins_df = pd.DataFrame(bins_data)

print(f'Expected Calibration Error (ECE): {ece:.4f}')
print(f'  (0 = perfectly calibrated, 1 = maximally miscalibrated)')

# Reliability diagram
fig, ax = penux_fig(8, 6, f'Reliability Diagram (ECE = {ece:.4f})')
non_empty = bins_df[bins_df['count'] > 0]
ax.bar(non_empty['bin_center'], non_empty['accuracy'], width=0.08,
       color=COLORS['teal'], alpha=0.8, edgecolor=COLORS['border'], label='Accuracy')
ax.plot([0, 1], [0, 1], '--', color=COLORS['muted'], linewidth=1, label='Perfect calibration')
ax.set_xlabel('Mean Predicted Confidence', color=COLORS['muted'])
ax.set_ylabel('Fraction Correct', color=COLORS['muted'])
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(facecolor=COLORS['panel'], edgecolor=COLORS['border'])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'reliability_diagram.png', dpi=150, bbox_inches='tight',
            facecolor=COLORS['bg'])
plt.show()

In [ ]:
# ── Subgroup Bias Analysis ──
# Evaluate model fairness across patient demographics (age groups, gender)

test_with_meta = pd.DataFrame(scaler.inverse_transform(X_test), columns=FEATURES)
test_with_meta['true'] = le.inverse_transform(y_test)
test_with_meta['pred'] = le.inverse_transform(y_pred)
test_with_meta['correct'] = (y_test == y_pred).astype(int)

# Add age groups
test_with_meta['age_group'] = pd.cut(test_with_meta['age'], bins=[0, 40, 60, 80, 120],
                                     labels=['18-39', '40-59', '60-79', '80+'])

print('=== Accuracy by Age Group ===')
age_bias = test_with_meta.groupby('age_group', observed=True)['correct'].agg(['mean', 'count'])
age_bias.columns = ['accuracy', 'n']
display(age_bias)

# Visualize
fig, ax = penux_fig(8, 5, 'Model Accuracy by Age Group')
bars = ax.bar(age_bias.index.astype(str), age_bias['accuracy'],
              color=COLORS['gold'], edgecolor=COLORS['border'])
ax.set_xlabel('Age Group', color=COLORS['muted'])
ax.set_ylabel('Accuracy', color=COLORS['muted'])
ax.set_ylim(0, 1)
for bar, acc, n in zip(bars, age_bias['accuracy'], age_bias['n']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{acc:.2f}\n(n={n})', ha='center', color=COLORS['text'], fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'bias_age_group.png', dpi=150, bbox_inches='tight',
            facecolor=COLORS['bg'])
plt.show()

## 7. Feature Signal Analysis

Visualize how each vital sign separates pathogen classes — the core clinical insight from penuX.

In [ ]:
# ── Distribution of each feature by pathogen class ──

fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor=COLORS['bg'])
label_colors = {'Normal': COLORS['teal'], 'Bacterial': COLORS['crimson'], 'Viral': COLORS['gold']}

for ax, feat, feat_label in zip(axes.flat, FEATURES, FEATURE_LABELS):
    ax.set_facecolor(COLORS['bg'])
    for label in sorted(dataset['label'].unique()):
        subset = dataset[dataset['label'] == label][feat].dropna()
        color = label_colors.get(label, COLORS['muted'])
        ax.hist(subset, bins=40, alpha=0.5, color=color, label=f'{label} (n={len(subset)})',
                edgecolor='none')
    ax.set_title(feat_label, color=COLORS['text'], fontsize=12, fontweight='bold')
    ax.set_xlabel(feat_label, color=COLORS['muted'])
    ax.set_ylabel('Count', color=COLORS['muted'])
    ax.legend(facecolor=COLORS['panel'], edgecolor=COLORS['border'], fontsize=8)
    for spine in ax.spines.values():
        spine.set_color(COLORS['border'])
    ax.tick_params(colors=COLORS['text'])

plt.suptitle('Feature Distributions by Pathogen Class', color=COLORS['text'],
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'feature_distributions.png', dpi=150, bbox_inches='tight',
            facecolor=COLORS['bg'])
plt.show()

In [ ]:
# ── Feature importance interpretation ──
# penuX priority ordering: Fever > WBC > SpO2 > Age

print('=== Clinical Feature Signal Strength (penuX interpretation) ===')
print()
print('1. FEVER (temperature_c)  — Primary discriminator')
print('   High fever + high WBC  -> Bacterial')
print('   High fever + normal WBC -> Viral')
print()
print('2. WBC COUNT              — Bacterial vs viral separation')
print('   Elevated (>10,000)     -> Suggests bacterial')
print('   Normal/low (4,000-10,000) -> Suggests viral')
print()
print('3. SpO2                   — Severity assessment')
print('   Low (<92%) + fever     -> Respiratory distress')
print()
print('4. AGE                    — Contextual factor')
print('   Older patients         -> Higher baseline risk')
print()
print('NOTE: These reflect empirical feature influence, not clinical rules.')

# Summary stats per class
print()
print('=== Mean Vitals by Class ===')
display(dataset.groupby('label')[FEATURES].mean().round(1))

## 8. Quick Recipes

In [ ]:
recipes = pd.DataFrame([
    {'task': 'Single prediction', 'code': 'predict_pathogen(temperature_c=38.5, wbc=14000, spo2=94, age=55)'},
    {'task': 'Batch prediction', 'code': 'predict_batch(my_df)'},
    {'task': 'Save predictions', 'code': "results.to_csv(OUTPUT_DIR / 'predictions.csv', index=False)"},
    {'task': 'Switch dataset', 'code': "SCHEMA = 'my_other_schema'  # then re-run extraction"},
    {'task': 'Overall correlation', 'code': "dataset[FEATURES].corr(method='pearson')"},
    {'task': 'Per-class correlation', 'code': "dataset[dataset['label']=='Bacterial'][FEATURES].corr()"},
    {'task': 'Calibration error', 'code': 'compute_ece(y_test, y_probs)'},
    {'task': 'Load saved model (TF)', 'code': "model = keras.models.load_model(OUTPUT_DIR / 'clin_encoder.keras')"},
    {'task': 'Export dataset', 'code': "dataset.to_csv(OUTPUT_DIR / 'clinical_dataset.csv', index=False)"},
])
recipes

---

## Next Steps

1. **Tune the model** — try different architectures, class weights, or add more features (lactate, respiratory rate)
2. **Species-level prediction** — use `org_name` directly as label for fine-grained pathogen classification
3. **Cross-dataset validation** — switch `SCHEMA` to another inpatient dataset and evaluate generalization
4. **Temporal analysis** — predict pathogen at different time windows within the admission
5. **Add imaging modality** — integrate chest X-ray data for the full penuX multimodal pipeline

**Citation:** Based on [penuX](https://github.com/netanelcyber/penuX) by netanelcyber.

**Research use only.** Model probabilities are not validated for clinical decision-making.